
# Jacobian Counter-example and Elliptic Integrals
## Executable companion to Artifact 24

This notebook follows the paper's organization. Narrative claims are preserved
as narrative; formulas that can be evaluated, checked, expanded, integrated,
or plotted are translated into Python.

Only bounded closed Abel-Wick cycles are plotted. Divergent real branches are
not part of this presentation.


In [ ]:

import numpy as np
import pandas as pd
import sympy as sp
import mpmath as mp
import plotly.io as pio

from artifact24.geometry import *
from artifact24.areas import *
from artifact24.plots import *

pio.renderers.default = "plotly_mimetype"
mp.mp.dps = 40



## 1. From the Jacobian conjecture to the triangle map

The polynomial map is \(F=(P,Q,R)\). The two-dimensional map studied in the
paper is \(G=F\circ\iota\), where \(\iota\) embeds the triangle plane.
The first executable check is that the three triangle corners collide.


In [ ]:

triangle_corners = np.array([[0,0], [1,0], [0,1]], dtype=float)
embedded_corners = triangle_embedding(triangle_corners[:,0], triangle_corners[:,1])
mapped_corners = polynomial_map(embedded_corners)

print("embedded corners:")
print(embedded_corners)
print("\nmapped corners:")
print(mapped_corners)
print("\nmaximum collision error:",
      np.max(np.linalg.norm(mapped_corners - COMMON_IMAGE, axis=1)))



## 2. The picture before the integrals

The triangle first integral is
\[
H(u,v)=uv(1-u-v),
\]
with maximum \(1/27\) at \((1/3,1/3)\). The centered cubic model is
\[
M(x,y)=x^2+y^2+\frac{2}{3\sqrt3}y(y^2-3x^2).
\]


In [ ]:

u, v = sp.symbols("u v", real=True)
H_symbolic = u*v*(1-u-v)
critical_solution = sp.solve(
    [sp.diff(H_symbolic,u), sp.diff(H_symbolic,v)],
    [u,v],
    dict=True,
)
critical_solution, sp.simplify(H_symbolic.subs(critical_solution[0]))



## 3. Two different meanings of area

- **Pre-image area:** flat area of the region before deformation.
- **Image area:** area measured along the mapped sheet, including local
  stretching by the polynomial map.

These are distinct computations and remain distinct in the code.



## 4. Integral I: pre-image area

The exact formula is
\[
\mathcal A_0(m)=\frac{2\pi\sqrt3}{27}S_3(m),\qquad
S_3(m)=m\,{}_2F_1\!\left(\frac13,\frac23;2;m\right).
\]


In [ ]:

sample_m = [0.1, 0.4, 0.7]
pd.DataFrame({
    "m": sample_m,
    "preimage_area": [float(preimage_area(m)) for m in sample_m],
    "preimage_action": [float(preimage_action(m)) for m in sample_m],
})


In [ ]:

n = sp.symbols("n", integer=True, nonnegative=True)
coefficients_A007004 = [
    sp.factorial(3*k) // ((k+1) * sp.factorial(k)**3)
    for k in range(10)
]
coefficients_A006480 = [
    sp.factorial(3*k) // (sp.factorial(k)**3)
    for k in range(10)
]
pd.DataFrame({
    "n": range(10),
    "A007004": coefficients_A007004,
    "A006480": coefficients_A006480,
})



## 5. Integral II: image area after stretching

The exact density is
\[
\mathcal J(x,y)=\|G_x\times G_y\|.
\]
The Python implementation derives \(G_x\times G_y\) symbolically from the
map, rather than copying the three long component polynomials by hand.


In [ ]:

cross_product = intrinsic_cross_product()
density_squared = intrinsic_density_squared()

print("G_x cross G_y:")
display(cross_product)
print("number of expanded terms in density squared:",
      len(sp.Poly(density_squared).terms()))
print("center density:", image_area_density(0.0, 0.0))
print("paper value sqrt(6267)/4:", J_CENTER)



The direct numerical integral is
\[
\mathcal A_\Sigma(m)=\iint_{M(x,y)\le m}\mathcal J(x,y)\,dx\,dy.
\]
The following cell evaluates it with Gaussian quadrature. It is intentionally
not executed automatically at the highest resolution on Binder.


In [ ]:

# Moderate Binder-safe resolution:
for m in [0.10, 0.40, 0.70]:
    value = image_area_quadrature(m, ntheta=80, nrho=36)
    print(f"m={m:.2f}: image area approx {value:.8f}")



## 6. Series and finite hypergeometric reductions

The exact coefficient tables and the \(N=8\) reduction are included as data
files. The original derivation scripts remain available, while this notebook
loads their outputs for inspection.


In [ ]:

series_table = pd.read_csv("data/true_surface_area_series.csv")
validation_table = pd.read_csv("data/true_surface_area_validation.csv")
display(series_table.head(10))
display(validation_table)



## 7. Closed cycles and exact matched levels

For every marked point on a principal axis, the notebook includes:

1. the red level curve at exactly that value of \(\alpha\);
2. the corresponding green, yellow, or blue bounded closed cycle at the same
   value of \(\alpha\);
3. their common source point;
4. its image under the polynomial map.


In [ ]:

fig = build_interactive_figure(
    red_count=12,
    auxiliary_count=8,
    samples=420,
    show_background=True,
)
fig.show(config={"scrollZoom": True, "displaylogo": False})



## 8. Folding the triangle continuously

The paper's final section poses an open geometric problem rather than a
completed computation: construct a smooth family \(G_t\) from the flat
triangle to the final mapped sheet, keeping the three corners distinct for
\(t<1\) and allowing their collision only in the limit.

No Python implementation is presented as a solution to that open problem.
A future notebook section may test candidate interpolations and monitor
Jacobian rank, corner separation, and mesh self-intersection.



## 9. Reproducibility

- Run all cells for the executable paper.
- Use the interactive figure directly in Binder.
- Use the Voilà entry point for an app-like presentation.
- The original PDF and TeX remain in `paper/`.
